# Exploratory Data Analysis (EDA)

This notebook covers the initial exploration of the uploaded ECG datasets: **MIT-BIH Arrhythmia**, **Chapman-Shaoxing Dataset**, and **PTB-XL**.

## Goals
1. Load and visualize raw ECG signals from both datasets.
2. Understand the data structure (sampling rate, leads, annotations).
3. Check for data quality and basic statistics.


In [ ]:
import pandas as pd
import numpy as np
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast

%matplotlib inline

## 1. Helper Functions

Functions to inspect records and plot signals.

In [ ]:
def print_record_info(record, title):
    print(f"--- {title} ---")
    print(f"Record Name: {record.record_name}")
    print(f"Signal Shape: {record.p_signal.shape}")
    print(f"Sampling Frequency: {record.fs} Hz")
    print(f"Signal Length: {record.sig_len} samples")
    print(f"Channel Names: {record.sig_name}")
    print(f"Comments: {record.comments}")
    print("\n")

def plot_sample_signal(record, title, channels=[0], limit=1000):
    plt.figure(figsize=(15, 5))
    for channel_idx in channels:
        if channel_idx < record.p_signal.shape[1]:
            channel_name = record.sig_name[channel_idx]
            plt.plot(record.p_signal[:limit, channel_idx], label=f'Lead {channel_name}')
    plt.title(f'{title} - First {limit} samples')
    plt.xlabel('Samples')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)
    plt.show()

## 2. MIT-BIH Arrhythmia Database Analysis

The MIT-BIH dataset contains 48 half-hour excerpts of two-channel ambulatory ECG recordings.
Path: `../data/raw/mit-bih/`

In [ ]:
mit_bih_path = '../data/raw/mit-bih/'
record_name = '100'  # Sample record

try:
    # Load record header and signal
    record_path = os.path.join(mit_bih_path, record_name)
    record_mit = wfdb.rdrecord(record_path)
    
    # Load annotations
    annotation_mit = wfdb.rdann(record_path, 'atr')
    
    print_record_info(record_mit, "MIT-BIH Record 100")
    plot_sample_signal(record_mit, "MIT-BIH Record 100", channels=[0, 1])
    
    # Plot with annotations
    print("Plotting signal with annotations...")
    wfdb.plot_wfdb(record=record_mit, annotation=annotation_mit,
                   title='MIT-BIH Record 100 with Annotations',
                   time_units='seconds',
                   figsize=(15, 6))

except Exception as e:
    print(f"Error loading MIT-BIH record: {e}")

## 3. Chapman-Shaoxing Dataset Analysis

The Chapman dataset contains 12-lead ECGs. 
Path: `../data/raw/chapman-shaoxing/`

In [ ]:
chapman_path = '../data/raw/chapman-shaoxing/'

# Locate a .hea file recursively since exact structure might vary
sample_record_path = None
for root, dirs, files in os.walk(chapman_path):
    for file in files:
        if file.endswith('.hea'):
            # Found a header file, use its base path (without extension)
            sample_record_path = os.path.join(root, os.path.splitext(file)[0])
            break
    if sample_record_path:
        break

if sample_record_path:
    try:
        print(f"Loading Chapman sample from: {sample_record_path}")
        record_chapman = wfdb.rdrecord(sample_record_path)
        
        print_record_info(record_chapman, "Chapman-Shaoxing Sample")
        
        # Plot subset of 12-leads (e.g., I, II, V1)
        # Lead indices: I=0, II=1, III=2, aVR=3, aVL=4, aVF=5, V1=6, V2=7, width...
        plot_sample_signal(record_chapman, "Chapman 12-Lead Sample (I, II, V1)", channels=[0, 1, 6])
        
    except Exception as e:
        print(f"Error loading Chapman record: {e}")
else:
    print("No .hea files found in Chapman directory path.")

## 4. PTB-XL Dataset Analysis

PTB-XL contains 21,837 12-lead ECG records. 
Path: `../data/raw/ptb-xl/`

In [ ]:
print("\n--- PTB-XL Dataset Analysis ---")
ptb_xl_path = '../data/raw/ptb-xl/'
csv_path = os.path.join(ptb_xl_path, 'ptbxl_database.csv')

# Try to find a sample record in records100 (low res) or records500 (high res)
sample_rec = 'records100/00000/00001_lr'
record_path = os.path.join(ptb_xl_path, sample_rec)

if os.path.exists(record_path + '.hea'):
    try:
        print(f"Loading PTB-XL sample: {sample_rec}")
        record_ptb = wfdb.rdrecord(record_path)
        print_record_info(record_ptb, "PTB-XL Record 00001_lr")
        plot_sample_signal(record_ptb, "PTB-XL 12-Lead (I, II, V1)", channels=[0, 1, 6])
    except Exception as e:
        print(f"Error loading PTB-XL record: {e}")
else:
    print(f"Sample PTB-XL record not found at {record_path}")

# Load Database Index for stats
if os.path.exists(csv_path):
    try:
        df_ptb = pd.read_csv(csv_path, index_col='ecg_id')
        df_ptb.scp_codes = df_ptb.scp_codes.apply(lambda x: ast.literal_eval(x))
        print(f"Total PTB-XL Records: {len(df_ptb)}")
        print("First 5 rows:")
        print(df_ptb[['patient_id', 'age', 'sex', 'scp_codes']].head())
    except Exception as e:
        print(f"Error loading PTB-XL database CSV: {e}")
else:
    print("ptbxl_database.csv not found.")